In [ ]:
# ============================================================
# EXPERIMENT 3: DATA PRE-PROCESSING
# Heart Disease Prediction
# ============================================================

# Import required libraries

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


In [ ]:
# ============================================================
# STEP 1: LOAD THE DATASET
# ============================================================

# Load the dataset
df = pd.read_csv("/content/Heart_disease_cleveland_new.csv")

print("First 5 rows:")
display(df.head())

print("\nDataset shape:")
print(df.shape)


In [ ]:
# ============================================================
# STEP 2: INSPECT THE DATASET
# ============================================================

print("Column names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nStatistical summary:")
display(df.describe())


In [ ]:
# ============================================================
# STEP 3: CHECK MISSING VALUES
# ============================================================

print("Missing values in each column:")
print(df.isnull().sum())

print("\nTotal missing values:")
print(df.isnull().sum().sum())


In [ ]:
# ============================================================
# STEP 4: CHECK AND REMOVE DUPLICATES
# ============================================================

duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

if duplicate_count > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")

print("Dataset shape after duplicate handling:")
print(df.shape)


In [ ]:
# ============================================================
# STEP 5: SEPARATE FEATURES AND TARGET
# ============================================================

# X contains input features
X = df.drop("target", axis=1)

# y contains the target variable
y = df["target"]

print("Features:")
print(X.columns.tolist())

print("\nFeature shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget values:")
print(y.value_counts())


In [ ]:
# ============================================================
# STEP 6: CHECK CLASS BALANCE
# ============================================================

class_counts = y.value_counts().sort_index()
class_percentages = y.value_counts(normalize=True).sort_index() * 100

class_balance = pd.DataFrame({
    "Count": class_counts,
    "Percentage": class_percentages.round(2)
})

print("Target class distribution:")
display(class_balance)

# The classes are reasonably balanced,
# so SMOTE/resampling is not required.


In [ ]:
# ============================================================
# STEP 7: DEFINE NUMERICAL AND CATEGORICAL FEATURES
# ============================================================

numerical_features = [
    "age",
    "trestbps",
    "chol",
    "thalach",
    "oldpeak"
]

categorical_features = [
    "sex",
    "cp",
    "fbs",
    "restecg",
    "exang",
    "slope",
    "ca",
    "thal"
]

print("Numerical features:")
print(numerical_features)

print("\nCategorical features:")
print(categorical_features)

print("\nTotal features:",
      len(numerical_features) + len(categorical_features))


In [ ]:
# ============================================================
# STEP 8: SPLIT DATA INTO TRAIN AND TEST
# ============================================================

# 80% of the data is used for training
# 20% of the data is kept for final testing
#
# stratify=y keeps the proportion of target classes
# approximately the same in both sets.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set shape:")
print(X_train.shape)

print("\nTest set shape:")
print(X_test.shape)


In [ ]:
# ============================================================
# STEP 9: CREATE PREPROCESSING PIPELINE
# ============================================================

# Numerical preprocessing
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

# Combine both preprocessing pipelines
preprocessor = ColumnTransformer([
    ("numerical", numerical_pipeline, numerical_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Preprocessing pipeline created.")


In [ ]:
# ============================================================
# STEP 10: FIT AND TRANSFORM THE DATA
# ============================================================

# Fit the preprocessing only on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Transform the test data using the fitted pipeline
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:")
print(X_train_processed.shape)

print("\nProcessed test shape:")
print(X_test_processed.shape)


In [ ]:
# ============================================================
# STEP 11: VERIFY PROCESSED DATA
# ============================================================

# Get names of all processed features
feature_names = preprocessor.get_feature_names_out()

# Convert processed data into DataFrames
X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names
)

# Reset indexes
X_train_processed_df = X_train_processed_df.reset_index(drop=True)
X_test_processed_df = X_test_processed_df.reset_index(drop=True)

y_train_processed = y_train.reset_index(drop=True)
y_test_processed = y_test.reset_index(drop=True)

print("First 5 rows of processed training data:")
display(X_train_processed_df.head())

print("\nNumber of processed features:")
print(X_train_processed_df.shape[1])

print("\nMissing values in processed training data:")
print(X_train_processed_df.isnull().sum().sum())

print("\nMissing values in processed test data:")
print(X_test_processed_df.isnull().sum().sum())

print("\nTraining class distribution:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTest class distribution:")
print(y_test.value_counts(normalize=True).round(3))


In [ ]:
# ============================================================
# STEP 12: SAVE PROCESSED DATA AND PIPELINE
# ============================================================

# Add target column to processed datasets
train_processed = X_train_processed_df.copy()
train_processed["target"] = y_train_processed

test_processed = X_test_processed_df.copy()
test_processed["target"] = y_test_processed

# Save processed datasets
train_processed.to_csv(
    "/content/heart_train_processed.csv",
    index=False
)

test_processed.to_csv(
    "/content/heart_test_processed.csv",
    index=False
)

# Save the preprocessing pipeline
joblib.dump(
    preprocessor,
    "/content/heart_preprocessor.pkl"
)

print("Files saved successfully:")
print("heart_train_processed.csv")
print("heart_test_processed.csv")
print("heart_preprocessor.pkl")


In [ ]:
# ============================================================
# FINAL CHECK
# ============================================================

print("============================================")
print("EXPERIMENT 3 PREPROCESSING COMPLETED")
print("============================================")

print("\nOriginal dataset:")
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nTrain samples:", len(X_train))
print("Test samples:", len(X_test))

print("\nProcessed features:", X_train_processed_df.shape[1])

print("\nMissing values after preprocessing:",
      X_train_processed_df.isnull().sum().sum())

print("\nDuplicate rows in processed training data:",
      X_train_processed_df.duplicated().sum())

print("\nClass distribution:")
print(y.value_counts())

print("\nPreprocessing pipeline saved successfully.")
